In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN, Dropout
import seaborn as sns
from google.colab import files
uploaded = files.upload()


Saving seattle-weather.csv to seattle-weather.csv


In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(df.head())
print(df.shape)

         date  precipitation  temp_max  temp_min  wind  weather
0  2012-01-01            0.0      12.8       5.0   4.7  drizzle
1  2012-01-02           10.9      10.6       2.8   4.5     rain
2  2012-01-03            0.8      11.7       7.2   2.3     rain
3  2012-01-04           20.3      12.2       5.6   4.7     rain
4  2012-01-05            1.3       8.9       2.8   6.1     rain
(1461, 6)


In [ ]:
# Cell 2
training_set = df.iloc[:,2:3].values

WINDOW = 10  # number of past days to use

def df_to_XY(training_set, window_size=WINDOW):
    X_train = []
    y_train = []
    for i in range(window_size, len(training_set)):
        X_train.append(training_set[i-window_size:i, 0])
        y_train.append(training_set[i, 0])
    X_train, y_train = np.array(X_train), np.array(y_train)
    return X_train, y_train

X, y = df_to_XY(training_set)

# train-test split
X_train = X[:800]
y_train = y[:800]
X_val = X[800:1000]
y_val = y[800:1000]
X_test = X[1000:]
y_test = y[1000:]

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_val = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

In [ ]:
# Cell 3
model = Sequential()

# Add SimpleRNN layers with Dropout
model.add(SimpleRNN(50, return_sequences=True, input_shape=(WINDOW,1)))
model.add(Dropout(0.2))

model.add(SimpleRNN(50, return_sequences=True))
model.add(Dropout(0.2))

model.add(SimpleRNN(50))
model.add(Dropout(0.2))

# Output layer
model.add(Dense(1))

# Compile model
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
# Cell 4
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)

In [ ]:
# Cell 5
his = pd.DataFrame(history.history)
plt.figure(figsize=(10,5))
plt.plot(his['loss'], label='Train Loss')
plt.plot(his['val_loss'], label='Validation Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Cell 6
train_pred = model.predict(X_train).flatten()
val_pred = model.predict(X_val).flatten()
test_pred = model.predict(X_test).flatten()

# Combine predictions for plotting
pred = np.concatenate([train_pred, val_pred, test_pred])
df_pred = pd.DataFrame(df["temp_max"].copy())
df_pred = df_pred[WINDOW:]
df_pred["predicted"] = pred

# Plot results
plt.figure(figsize=(14,6))
plt.plot(df_pred[800:]["predicted"], label="Predicted")
plt.plot(df_pred[800:]["temp_max"], label="Actual")
plt.title("Validation & Test Predictions")
plt.xlabel("Day")
plt.ylabel("Temp Max")
plt.legend()
plt.show()
df_pred[800:].head(10)  # Shows first 10 actual vs predicted values

In [ ]:
df['date'] = pd.to_datetime(df['date'])

# Take user input
user_date_str = input("Enter a date (YYYY-MM-DD): ")
user_date = pd.to_datetime(user_date_str)

# Check if the date exists in the dataset
if user_date not in df['date'].values:
    print("Date not found in dataset.")
else:
    idx = df.index[df['date'] == user_date][0]

    # Make sure there are at least 10 previous days
    if idx < WINDOW:
        print("Not enough previous data to make a prediction.")
    else:
        last_10_days = df['temp_max'].iloc[idx-WINDOW:idx].values
        last_10_days = last_10_days.reshape(1, WINDOW, 1)

        # Predict next day
        pred_temp = model.predict(last_10_days)
        print(f"Predicted max temperature for {user_date + pd.Timedelta(days=1)}: {pred_temp[0][0]:.2f} °C")